<a href="https://colab.research.google.com/github/rsher60/LLM_Codebase/blob/main/Chunker_streamlit_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import base64
import asyncio
import io
import json
import re
from typing import List, Dict, Optional
from datetime import datetime
import fitz
from PIL import Image

import numpy as np
import streamlit as st
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.docstore.document import Document
from langchain_openai import OpenAIEmbeddings
from langchain.schema import HumanMessage
from langchain.text_splitter import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)
from sklearn.metrics.pairwise import cosine_similarity

# Load environment variables
load_dotenv()

# --- Constants and Configurations ---
DEFAULT_IMAGE_PROMPT = """
Extract ALL text content from this image page and format it as clean, structured Markdown.

**Instructions:**
1. Preserve the original document structure and hierarchy
2. Use appropriate Markdown headers (# ## ### ####) for titles and sections
3. Convert tables to Markdown table format
4. Preserve lists as Markdown lists (- or 1.)
5. Keep code blocks in proper code fences if present
6. Include ALL visible text - don't summarize or omit content
7. Maintain proper spacing and paragraph breaks
8. If there are images/diagrams, describe them briefly in italic text

**Format the output as clean, readable Markdown that preserves the document's structure.**
"""

OPENAI_MODEL_OPTIONS = ["gpt-4o", "gpt-4-turbo", "gpt-4"]
DEFAULT_CHUNK_SIZE = 1000
DEFAULT_CHUNK_OVERLAP = 200
DEFAULT_SIMILARITY_THRESHOLD = 0.7

# MongoDB connection code


from pymongo import MongoClient


def connect_to_mongodb(uri: str, db_name: str, collection_name: str, chunks):
    """Connect to MongoDB and return the collection."""
    client = MongoClient(uri)
    db = client[db_name]
    collection = db[collection_name]
    result = collection.insert_many(chunks)
    client.close()
    return (f"Inserted {len(result.inserted_ids)} documents into the collection.")






# --- Helper Classes ---
class HeaderTracker:
    """Tracks markdown headers and their hierarchy for chunk mapping"""

    def __init__(self):
        self.header_stack = []

    def extract_headers_from_text(self, text: str) -> List[Dict]:
        """Extract all headers from markdown text with their positions"""
        headers = []
        lines = text.split("\n")

        for i, line in enumerate(lines):
            line = line.strip()
            if line.startswith("#"):
                level = len(line) - len(line.lstrip("#"))
                if level <= 6:
                    header_text = line.lstrip("#").strip()
                    headers.append(
                        {
                            "level": level,
                            "text": header_text,
                            "line_number": i,
                            "full_header": line,
                        }
                    )
        return headers

    def get_header_hierarchy_for_position(
        self, headers: List[Dict], target_line: int
    ) -> Dict[str, str]:
        """Get the current header hierarchy for a specific line position"""
        hierarchy = {}
        current_stack = []

        for header in headers:
            if header["line_number"] > target_line:
                break

            level = header["level"]

            current_stack = [h for h in current_stack if h["level"] < level]
            current_stack.append(header)

        for header in current_stack:
            hierarchy[f"Header {header['level']}"] = header["text"]
        return hierarchy

    def get_comprehensive_header_mapping(
        self, headers: List[Dict], target_line: int
    ) -> Dict[str, any]:
        """Get comprehensive header mapping including parent-child relationships"""
        hierarchy = {}
        current_stack = []

        # Build the current stack for the target position
        for header in headers:
            if header["line_number"] > target_line:
                break

            level = header["level"]
            # Remove headers at same or deeper level
            current_stack = [h for h in current_stack if h["level"] < level]
            current_stack.append(header)

        # Create the basic header hierarchy
        for header in current_stack:
            hierarchy[f"Header {header['level']}"] = header["text"]

        # Create comprehensive mapping
        comprehensive_mapping = {
            "header_hierarchy": hierarchy,
            "header_path": [h["text"] for h in current_stack],  # Ordered path from root to current
            #"current_headers": {
               # "h1": None, "h2": None, "h3": None, "h4": None, "h5": None, "h6": None
            #},
            "parent_child_mapping": {}
        }

        # Fill current headers
        #for header in current_stack:
            #comprehensive_mapping["current_headers"][f"h{header['level']}"] = header["text"]

        # Create parent-child relationships
        for i, header in enumerate(current_stack):
            if i > 0:
                parent = current_stack[i-1]
                comprehensive_mapping["parent_child_mapping"][header["text"]] = parent["text"]

        return comprehensive_mapping


class SemanticChunker:
    """Semantic chunking based on content similarity using embeddings"""

    def __init__(self, openai_api_key: str, similarity_threshold: float):
        self.embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)
        self.similarity_threshold = similarity_threshold
        self.header_tracker = HeaderTracker()

    def create_embeddings(self, texts: List[str]) -> np.ndarray:
        """Create embeddings for a list of texts"""
        embeddings = self.embeddings.embed_documents(texts)
        return np.array(embeddings)

    def _split_into_sentences_with_positions(self, text: str) -> List[Dict]:
        """Split text into sentences/paragraphs with line position tracking"""
        lines = text.split("\n")
        sentences = []

        for line_num, line in enumerate(lines):
            line = line.strip()
            if not line:
                continue

            if line.startswith("#"):
                continue

            if len(line) > 200:
                sent_splits = re.split(r"[.!?]+\s+", line)
                for sent in sent_splits:
                    if sent.strip():
                        sentences.append({"text": sent.strip(), "line_number": line_num})
            else:
                sentences.append({"text": line, "line_number": line_num})
        return sentences

    def split_text_semantically_with_headers(
        self, text: str, max_chunk_size: int = DEFAULT_CHUNK_SIZE
    ) -> List[Document]:
        """Split text based on semantic similarity while preserving header mapping"""
        headers = self.header_tracker.extract_headers_from_text(text)
        sentences_with_positions = self._split_into_sentences_with_positions(text)

        if len(sentences_with_positions) <= 1:
            comprehensive_mapping = self.header_tracker.get_comprehensive_header_mapping(headers, 0)
            return [
                Document(
                    page_content=text,
                    metadata={
                        "chunk_type": "semantic",
                        "chunk_id": 0,
                        **comprehensive_mapping,
                    },
                )
            ]

        embeddings = self.create_embeddings([item["text"] for item in sentences_with_positions])

        chunks = []
        current_chunk = [sentences_with_positions[0]]
        current_embedding = embeddings[0:1]

        for i in range(1, len(sentences_with_positions)):
            chunk_avg_embedding = np.mean(current_embedding, axis=0).reshape(1, -1)
            sentence_embedding = embeddings[i : i + 1]

            similarity = cosine_similarity(chunk_avg_embedding, sentence_embedding)[0][0]

            potential_chunk_texts = [
                item["text"] for item in current_chunk + [sentences_with_positions[i]]
            ]
            potential_chunk_size = len(" ".join(potential_chunk_texts))

            if similarity > self.similarity_threshold and potential_chunk_size <= max_chunk_size:
                current_chunk.append(sentences_with_positions[i])
                current_embedding = np.vstack([current_embedding, sentence_embedding])
            else:
                chunks.append(current_chunk)
                current_chunk = [sentences_with_positions[i]]
                current_embedding = sentence_embedding

        if current_chunk:
            chunks.append(current_chunk)

        documents = []
        for i, chunk_sentences in enumerate(chunks):
            chunk_text = " ".join([item["text"] for item in chunk_sentences])
            first_sentence_line = chunk_sentences[0]["line_number"]
            comprehensive_mapping = self.header_tracker.get_comprehensive_header_mapping(
                headers, first_sentence_line
            )

            doc = Document(
                page_content=chunk_text,
                metadata={
                    "chunk_type": "semantic",
                    "chunk_id": i,
                    "total_chunks": len(chunks),
                    "start_line": chunk_sentences[0]["line_number"],
                    "end_line": chunk_sentences[-1]["line_number"],
                    **comprehensive_mapping,
                },
            )
            documents.append(doc)
        return documents


class CombinedChunker:
    """Combines Markdown Header Text Splitter with Semantic Chunker"""

    def __init__(self, openai_api_key: str, similarity_threshold: float):
        self.semantic_chunker = SemanticChunker(openai_api_key, similarity_threshold)
        self.header_tracker = HeaderTracker()

    def chunk_with_combined_approach(
        self,
        markdown_content: str,
        chunk_size: int = DEFAULT_CHUNK_SIZE,
        chunk_overlap: int = DEFAULT_CHUNK_OVERLAP,
        similarity_threshold: float = DEFAULT_SIMILARITY_THRESHOLD,
    ) -> List[Document]:
        """
        Use Markdown Header Text Splitter first, then apply semantic chunking to large sections
        """
        headers_to_split_on = [
            ("#", "Header 1"),
            ("##", "Header 2"),
            ("###", "Header 3"),
            ("####", "Header 4"),
        ]

        markdown_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=headers_to_split_on, strip_headers=False
        )
        md_header_splits = markdown_splitter.split_text(markdown_content)

        final_chunks = []
        headers = self.header_tracker.extract_headers_from_text(markdown_content)

        for doc in md_header_splits:
            # Extract position information to get comprehensive header mapping
            content_lines = markdown_content.split('\n')
            content_start_line = 0
            for i, line in enumerate(content_lines):
                if doc.page_content.strip().startswith(line.strip()) and line.strip():
                    content_start_line = i
                    break

            comprehensive_mapping = self.header_tracker.get_comprehensive_header_mapping(
                headers, content_start_line
            )

            enhanced_metadata = doc.metadata.copy()

            if len(doc.page_content) > chunk_size:
                self.semantic_chunker.similarity_threshold = similarity_threshold
                semantic_chunks = self.semantic_chunker.split_text_semantically_with_headers(
                    doc.page_content, chunk_size
                )

                for semantic_chunk in semantic_chunks:
                    # Merge the comprehensive mappings
                    merged_mapping = comprehensive_mapping.copy()
                    chunk_mapping = {
                        k: v for k, v in semantic_chunk.metadata.items()
                        #if k in ["header_hierarchy", "header_path", "current_headers", "parent_child_mapping"]
                        if k in ["header_hierarchy", "header_path", "parent_child_mapping"]
                    }

                    # Merge header hierarchies
                    if "header_hierarchy" in chunk_mapping:
                        merged_hierarchy = merged_mapping["header_hierarchy"].copy()
                        merged_hierarchy.update(chunk_mapping["header_hierarchy"])
                        merged_mapping["header_hierarchy"] = merged_hierarchy

                    semantic_chunk.metadata.update(enhanced_metadata)
                    semantic_chunk.metadata.update(merged_mapping)
                    semantic_chunk.metadata["chunk_type"] = "combined_markdown_semantic"

                final_chunks.extend(semantic_chunks)
            else:
                doc.metadata.update(comprehensive_mapping)
                doc.metadata["chunk_type"] = "markdown_header_only"
                final_chunks.append(doc)

        for i, chunk in enumerate(final_chunks):
            chunk.metadata.update({"chunk_id": i, "total_chunks": len(final_chunks)})
        return final_chunks


class PDFToMarkdownProcessor:
    """Streamlit-optimized PDF to Markdown processor"""

    def __init__(self, openai_api_key: str, model_name: str = "gpt-4o"):
        self.openai_api_key = openai_api_key
        self.model_name = model_name
        self.llm = ChatOpenAI(
            model=model_name,
            openai_api_key=openai_api_key,
            max_tokens=4096,
            temperature=0.1,
        )
        self.semantic_chunker = SemanticChunker(openai_api_key, DEFAULT_SIMILARITY_THRESHOLD)
        self.combined_chunker = CombinedChunker(openai_api_key, DEFAULT_SIMILARITY_THRESHOLD)
        self.header_tracker = HeaderTracker()
        self.extraction_prompt = DEFAULT_IMAGE_PROMPT

    def pdf_to_images(self, pdf_bytes: bytes, dpi: int = 200) -> List[Image.Image]:
        """Convert PDF bytes to images"""
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        images = []
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            mat = fitz.Matrix(dpi / 72, dpi / 72)
            pix = page.get_pixmap(matrix=mat, alpha=False)
            img_data = pix.tobytes("png")
            img = Image.open(io.BytesIO(img_data))
            images.append(img)
        doc.close()
        return images

    def image_to_base64(self, image: Image.Image) -> str:
        """Convert PIL Image to base64 string"""
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=85, optimize=True)
        return base64.b64encode(buffer.getvalue()).decode()

    async def extract_content_from_image(self, image: Image.Image, page_num: int) -> str:
        """Extract content from image using OpenAI Vision API"""
        try:
            base64_image = self.image_to_base64(image)
            message = HumanMessage(
                content=[
                    {"type": "text", "text": f"{self.extraction_prompt}\n\n**Page {page_num + 1}:**"},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{base64_image}", "detail": "high"},
                    },
                ]
            )
            response = await self.llm.ainvoke([message])
            return response.content
        except Exception as e:
            return f"# Page {page_num + 1}\n\n*Error processing this page: {str(e)}*\n\n"

    async def process_pdf_async(self, pdf_bytes: bytes, progress_callback=None) -> str:
        """Process PDF bytes to markdown with progress updates"""
        images = self.pdf_to_images(pdf_bytes)

        if progress_callback:
            progress_callback(0.2, f"Converting PDF to {len(images)} images...")

        semaphore = asyncio.Semaphore(3)  # Limit concurrent API calls

        async def process_page_with_limit(img, page_num):
            async with semaphore:
                result = await self.extract_content_from_image(img, page_num)
                if progress_callback:
                    progress = 0.2 + (0.6 * (page_num + 1) / len(images))
                    progress_callback(progress, f"Processed page {page_num + 1}/{len(images)}")
                return result

        tasks = [process_page_with_limit(img, i) for i, img in enumerate(images)]
        page_contents = await asyncio.gather(*tasks, return_exceptions=True)

        full_markdown = ""
        for i, content in enumerate(page_contents):
            if isinstance(content, Exception):
                full_markdown += f"\n\n# Page {i + 1}\n\n*Error: {str(content)}*\n\n"
            else:
                full_markdown += f"\n\n{content}\n\n"

        if progress_callback:
            progress_callback(0.9, "Cleaning up markdown...")

        full_markdown = self._clean_markdown(full_markdown)
        return full_markdown

    def _clean_markdown(self, markdown_content: str) -> str:
        """Clean and optimize markdown content"""
        lines = markdown_content.split("\n")
        cleaned_lines = []
        for line in lines:
            line = line.strip()
            if line == "" and len(cleaned_lines) > 0 and cleaned_lines[-1] == "":
                continue
            cleaned_lines.append(line)
        return "\n".join(cleaned_lines)

    def chunk_markdown_with_headers(
        self, markdown_content: str, chunk_size: int, chunk_overlap: int
    ) -> List[Document]:
        """Chunk markdown content using MarkdownHeaderTextSplitter with comprehensive header mapping"""
        headers_to_split_on = [
            ("#", "Header 1"),
            ("##", "Header 2"),
            ("###", "Header 3"),
            ("####", "Header 4"),
        ]

        markdown_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=headers_to_split_on, strip_headers=False
        )
        md_header_splits = markdown_splitter.split_text(markdown_content)

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=["\n\n", "\n", " ", ""]
        )

        final_chunks = []
        headers = self.header_tracker.extract_headers_from_text(markdown_content)

        for doc in md_header_splits:
            # Find the position of this content in the original markdown
            content_lines = markdown_content.split('\n')
            content_start_line = 0
            for i, line in enumerate(content_lines):
                if doc.page_content.strip().startswith(line.strip()) and line.strip():
                    content_start_line = i
                    break

            comprehensive_mapping = self.header_tracker.get_comprehensive_header_mapping(
                headers, content_start_line
            )

            if len(doc.page_content) > chunk_size:
                sub_chunks = text_splitter.split_documents([doc])
                for sub_chunk in sub_chunks:
                    sub_chunk.metadata.update(comprehensive_mapping)
                    sub_chunk.metadata["chunk_type"] = "markdown_header_recursive"
                final_chunks.extend(sub_chunks)
            else:
                doc.metadata.update(comprehensive_mapping)
                doc.metadata["chunk_type"] = "markdown_header"
                final_chunks.append(doc)

        for i, chunk in enumerate(final_chunks):
            chunk.metadata.update({"chunk_id": i, "total_chunks": len(final_chunks)})
        return final_chunks

    def chunk_with_semantic_chunker(
        self, markdown_content: str, chunk_size: int, similarity_threshold: float
    ) -> List[Document]:
        """Chunk using semantic similarity with comprehensive header mapping"""
        self.semantic_chunker.similarity_threshold = similarity_threshold
        return self.semantic_chunker.split_text_semantically_with_headers(
            markdown_content, chunk_size
        )

    def chunk_with_combined_approach(
        self,
        markdown_content: str,
        chunk_size: int,
        chunk_overlap: int,
        similarity_threshold: float,
    ) -> List[Document]:
        """Use combined markdown + semantic chunking approach with comprehensive header mapping"""
        return self.combined_chunker.chunk_with_combined_approach(
            markdown_content, chunk_size, chunk_overlap, similarity_threshold
        )
# --- Streamlit App ---
def main():


    MONGODB_URI = os.getenv("MONGODB_URI")
    COLLECTION_NAME = os.getenv("MONGODB_COLLECTION")
    DATABASE_NAME = os.getenv("MONGODB_DATABASE")

    #print(f"MongoDB URI: {MONGODB_URI}")
    #print(f"MongoDB Collection: {COLLECTION_NAME}")
    #print(f"MongoDB Database: {DATABASE_NAME}")


    st.set_page_config(
        page_title="PDF to Markdown Chunker with Header Mapping",
        page_icon="📄",
        layout="wide",
        initial_sidebar_state="expanded",
    )

    st.title("📄 PDF to Markdown Chunker with Header Mapping")
    st.markdown("Convert PDFs to Markdown and chunk them using different strategies with parent header tracking")

    # --- Sidebar Configuration ---
    st.sidebar.header("⚙️ Configuration")

    api_key = st.sidebar.text_input(
        "OpenAI API Key",
        type="password",
        help="Enter your OpenAI API key",
        value=os.getenv("OPENAI_API_KEY", ""),
    )

    if not api_key:
        st.warning("Please enter your OpenAI API key in the sidebar to continue.")
        st.stop()

    chunking_method = st.sidebar.selectbox(
        "Chunking Method",
        ["Markdown Header Text Splitter", "Semantic Chunker", "Combined (Markdown + Semantic)"],
        index=2,
        help="Choose the chunking strategy",
    )

    st.sidebar.subheader("Chunking Parameters")
    chunk_size = st.sidebar.slider(
        "Chunk Size",
        min_value=200,
        max_value=3000,
        value=DEFAULT_CHUNK_SIZE,
        step=100,
        help="Maximum size of each chunk in characters",
    )

    chunk_overlap = DEFAULT_CHUNK_OVERLAP
    if chunking_method in ["Markdown Header Text Splitter", "Combined (Markdown + Semantic)"]:
        chunk_overlap = st.sidebar.slider(
            "Chunk Overlap",
            min_value=0,
            max_value=500,
            value=DEFAULT_CHUNK_OVERLAP,
            step=50,
            help="Overlap between consecutive chunks",
        )

    similarity_threshold = DEFAULT_SIMILARITY_THRESHOLD
    if chunking_method in ["Semantic Chunker", "Combined (Markdown + Semantic)"]:
        similarity_threshold = st.sidebar.slider(
            "Similarity Threshold",
            min_value=0.5,
            max_value=1.0,
            value=DEFAULT_SIMILARITY_THRESHOLD,
            step=0.05,
            help="Minimum similarity to group sentences together",
        )

    model_name = st.sidebar.selectbox(
        "OpenAI Model", OPENAI_MODEL_OPTIONS, index=0, help="Choose the OpenAI model for vision processing"
    )

    # --- Main Content Area ---
    col1, col2 = st.columns([1, 1])

    with col1:
        st.header("📤 Upload PDF")
        uploaded_file = st.file_uploader(
            "Choose a PDF file",
            type="pdf",
            help="Upload a PDF file to convert to markdown and chunk",
            key="pdf_uploader",  # Add a key to the uploader
        )

        if uploaded_file and "last_uploaded_file_id" not in st.session_state:
            st.session_state.last_uploaded_file_id = uploaded_file.file_id
            st.session_state.clear_results = True  # Flag to clear previous results

        # Clear results if a new file is uploaded
        if uploaded_file and st.session_state.get("last_uploaded_file_id") != uploaded_file.file_id:
            st.session_state.clear_results = True
            st.session_state.last_uploaded_file_id = uploaded_file.file_id

        if st.session_state.get("clear_results", False):
            if "markdown_content" in st.session_state:
                del st.session_state["markdown_content"]
            if "chunks" in st.session_state:
                del st.session_state["chunks"]
            st.session_state.clear_results = False

        if uploaded_file:
            st.success(f"Uploaded: {uploaded_file.name}")
            st.info(f"File size: {len(uploaded_file.getvalue()) / (1024 * 1024):.2f} MB")

    with col2:
        st.header("📊 Processing Status")
        status_container = st.empty()
        progress_container = st.empty()

    # Process button
    if uploaded_file and st.button("🚀 Process PDF", type="primary"):
        # Ensure only one processing run is active
        if st.session_state.get("processing_in_progress", False):
            st.warning("Processing is already in progress. Please wait.")
            return

        st.session_state.processing_in_progress = True
        try:
            processor = PDFToMarkdownProcessor(openai_api_key=api_key, model_name=model_name)

            progress_bar = progress_container.progress(0)
            status_text = status_container.empty()

            def update_progress(progress_val: float, message: str):
                progress_bar.progress(progress_val)
                status_text.text(message)

            pdf_bytes = uploaded_file.getvalue()

            # Use st.spinner for long operations instead of asyncio.run directly
            with st.spinner("Starting PDF processing..."):
                # Run the async function using a custom event loop or just let Streamlit handle it
                # For simplicity and Streamlit compatibility, we'll avoid asyncio.run() here
                # and call the async function directly. Streamlit's internal event loop
                # often handles this implicitly for async functions called from sync context
                # within its own execution model.
                markdown_content = asyncio.run(
                    processor.process_pdf_async(pdf_bytes, progress_callback=update_progress)
                )

            update_progress(0.95, "Chunking content...")

            chunks = []
            if chunking_method == "Markdown Header Text Splitter":
                chunks = processor.chunk_markdown_with_headers(
                    markdown_content, chunk_size=chunk_size, chunk_overlap=chunk_overlap
                )
            elif chunking_method == "Semantic Chunker":
                chunks = processor.chunk_with_semantic_chunker(
                    markdown_content, chunk_size=chunk_size, similarity_threshold=similarity_threshold
                )
            else:  # Combined approach
                chunks = processor.chunk_with_combined_approach(
                    markdown_content,
                    chunk_size=chunk_size,
                    chunk_overlap=chunk_overlap,
                    similarity_threshold=similarity_threshold,
                )

            update_progress(1.0, "Processing complete!")

            st.session_state.markdown_content = markdown_content
            st.session_state.chunks = chunks
            st.session_state.chunking_method = chunking_method

            st.success(f"✅ Processing complete! Generated {len(chunks)} chunks.")

        except Exception as e:
            st.error(f"❌ Error processing PDF: {str(e)}")
        finally:
            st.session_state.processing_in_progress = False

    # --- Display Results ---
    if "markdown_content" in st.session_state and st.session_state.markdown_content:
        st.header("📋 Results")

        tab1, tab2, tab3, tab4 = st.tabs(["📄 Markdown Content", "🔗 Chunks", "📊 Statistics", "🏷️ Header Mapping"])

        with tab1:
            st.subheader("Generated Markdown")
            st.code(st.session_state.markdown_content , language="markdown")

            st.download_button(
                label="📥 Download Markdown",
                data=st.session_state.markdown_content,
                file_name=f"{uploaded_file.name.replace('.pdf', '.md') if uploaded_file else 'output.md'}",
                mime="text/markdown",
            )

        with tab2:
            st.subheader(f"Chunks ({st.session_state.chunking_method})")

            for i, chunk in enumerate(st.session_state.chunks):
                with st.expander(f"Chunk {i+1} ({len(chunk.page_content)} chars)"):
                    st.markdown("**Metadata:**")
                    st.json(chunk.metadata)
                    st.markdown("** Header + Content**")
                    st.markdown(str(chunk.metadata.get("header_hierarchy", {})) + chunk.page_content )

            chunks_data = [
                {
                    "chunk_id": i,
                    "content": chunk.page_content,
                    "metadata": chunk.metadata,
                    "length": len(chunk.page_content),
                }
                for i, chunk in enumerate(st.session_state.chunks)
            ]

            #combined = chunk.page_content |  chunk.metadata
            #combined.update(chunk.metadata) # to pass this to MongoDb if user asks for it



            def connect_to_mongodb(uri: str, db_name: str, collection_name: str, chunks):
                """Connect to MongoDB and insert documents."""
                client = MongoClient(MONGODB_URI)
                db = client[DATABASE_NAME]
                collection = db[COLLECTION_NAME]
                result = collection.insert_many(chunks)
                client.close()
                return f"Inserted {len(result.inserted_ids)} documents into the collection."



            col1, col2 = st.columns(2)

            with col1:
                st.download_button(
                    label="📥 Download Chunks (JSON)",
                    data=json.dumps(chunks_data, indent=2),
                    file_name=f"{uploaded_file.name.replace('.pdf', '_chunks.json') if uploaded_file else 'chunks.json'}",
                    mime="application/json",
                )

            with col2:
                if st.button(
                    label="Send Chunks to Vector Store",
                    help="Send the chunks to your vector store for further processing or querying.",
                ):
                    if 'chunks' in st.session_state and st.session_state.chunks:
                        # MongoDB configuration for your specific setup
                        #MONGODB_URI = "mongodb://localhost:27017/"  # Replace with your actual MongoDB URI
                        #DATABASE_NAME = "user_documents"             # Your database name
                        #COLLECTION_NAME = "docs"                     # Your collection name


                        MONGODB_URI = os.getenv("MONGODB_URI")
                        DATABASE_NAME = os.getenv("MONGODB_DATABASE")
                        COLLECTION_NAME = os.getenv("MONGODB_COLLECTION")



                        try:
                            # Prepare documents for MongoDB insertion
                            documents_to_insert = []

                            for i, chunk in enumerate(st.session_state.chunks):
                                # Create document with proper structure
                                document = {
                                    "data": chunk.page_content,           # Main content
                                    "metadata": chunk.metadata,           # Chunk metadata
                                    "chunk_index": i,                     # Index for ordering
                                    "inserted_at": datetime.now(),        # Timestamp
                                    "document_id": chunk.metadata.get('source', f'doc_{i}'),  # Document identifier
                                    "page_number": chunk.metadata.get('page', None),          # Page number if available
                                    "document_type": "text_chunk"         # Type identifier
                                }
                                documents_to_insert.append(document)

                            # Insert documents into MongoDB
                            result_message = connect_to_mongodb(
                                uri=MONGODB_URI,
                                db_name=DATABASE_NAME,
                                collection_name=COLLECTION_NAME,
                                chunks=documents_to_insert
                            )

                            st.success(result_message)
                            st.info(f"Documents saved to database: '{DATABASE_NAME}', collection: '{COLLECTION_NAME}'")

                            # Optional: Show sample of what was inserted
                            with st.expander("View Sample Inserted Document"):
                                st.json(documents_to_insert[0] if documents_to_insert else {})

                        except Exception as e:
                            st.error(f"Error connecting to MongoDB: {str(e)}")
                            # Additional error details for debugging
                            import traceback
                            st.error(f"Detailed error: {traceback.format_exc()}")

                    else:
                        st.warning("No chunks available to send. Please process your document first.")



        with tab3:
            st.subheader("Processing Statistics")

            chunks = st.session_state.chunks
            chunk_lengths = [len(chunk.page_content) for chunk in chunks]

            col1, col2, col3 = st.columns(3)

            with col1:
                st.metric("Total Chunks", len(chunks))
                st.metric("Average Chunk Size", f"{np.mean(chunk_lengths):.0f} chars")

            with col2:
                st.metric("Min Chunk Size", f"{min(chunk_lengths)} chars")
                st.metric("Max Chunk Size", f"{max(chunk_lengths)} chars")

            with col3:
                st.metric("Total Content Length", f"{len(st.session_state.markdown_content)} chars")
                st.metric("Chunking Method", st.session_state.chunking_method)

            st.subheader("Chunk Size Distribution")
            st.bar_chart(chunk_lengths)

        with tab4:
            st.subheader("Header Mapping Analysis")

            chunks_with_headers = [
                chunk for chunk in st.session_state.chunks if chunk.metadata.get("header_hierarchy")
            ]

            if chunks_with_headers:
                st.info(f"Found {len(chunks_with_headers)} chunks with header mappings")

                for i, chunk in enumerate(chunks_with_headers):
                    hierarchy = chunk.metadata.get("header_hierarchy", {})
                    if hierarchy:
                        with st.expander(f"Chunk {chunk.metadata.get('chunk_id', i)+1} - Header Hierarchy"):
                            st.markdown("**Header Path:**")
                            for level in sorted(hierarchy.keys()):
                                indent = "  " * (int(level.split()[-1]) - 1)
                                st.markdown(f"{indent}- {level}: {hierarchy[level]}")

                            st.markdown("**Chunk Preview:**")
                            preview = (
                                chunk.page_content[:200] + "..."
                                if len(chunk.page_content) > 200
                                else chunk.page_content
                            )
                            st.code(preview)

                st.subheader("Header Distribution Summary")
                header_counts = {}
                for chunk in chunks_with_headers:
                    hierarchy = chunk.metadata.get("header_hierarchy", {})
                    for level, header_text in hierarchy.items():
                        if level not in header_counts:
                            header_counts[level] = {}
                        if header_text not in header_counts[level]:
                            header_counts[level][header_text] = 0
                        header_counts[level][header_text] += 1

                for level in sorted(header_counts.keys()):
                    st.markdown(f"**{level} Distribution:**")
                    for header, count in header_counts[level].items():
                        st.markdown(f"  - {header}: {count} chunks")
            else:
                st.warning(
                    "No header mappings found in chunks. This might indicate that the markdown doesn't contain headers or the chunking method doesn't preserve header information."
                )


if __name__ == "__main__":
    main()